# Milestone 9b — GPT-5.2 with the selected context-engineered system

## Goal

Complete the DomainAdapt benchmark matrix by applying the exact Milestone 5 context-engineering contract to GPT-5.2.

Earlier experiments measured:

- GPT-5.2 with minimal schema context;
- bare Ministral-3B;
- context-engineered Ministral-3B; and
- fine-tuned Ministral-3B with that selected context.

The missing comparison is GPT-5.2 with the same selected system. This experiment isolates whether context engineering merely compensates for a weak model or also adds value to a stronger model.

No prompt or context optimization occurs here. GPT-5.2 receives the already frozen compact context, guardrails, verified values, and strict output normalization unchanged.


## Research question and possible outcomes

> How does the selected context-engineering system change GPT-5.2 execution accuracy relative to the original minimal-context frontier baseline?

| Outcome | Interpretation |
|---|---|
| Large improvement | Frontier capability and domain context are complementary |
| Small improvement | Context helps, but most performance comes from the stronger model |
| No change | GPT-5.2 already inferred most information supplied by the context |
| Regression | The additional context or prescriptive guardrails distract the stronger model |

This is an architectural ablation, not another adaptation attempt. Its value comes from changing one independent variable: the system context around GPT-5.2.


## Controlled experiment contract

| Component | Fixed value |
|---|---|
| Model | GPT-5.2 deployment, version `2025-12-11` |
| API | Azure OpenAI Responses API |
| Instructions | Selected Milestone 5 guardrails |
| Schema context | Selected compact financial context with meanings and verified values |
| Prompt format | Existing `Database schema` + `Question` format |
| Output normalization | Strict removal of one surrounding SQL fence |
| Attempts | One valid response per question |
| Candidate repair | Disabled |
| Incorrect-answer retry | Disabled |
| Evaluation | Same SQLite execution comparator and reference-only compatibility rules |

GPT-5.2 uses the Responses API while Ministral-3B uses Chat Completions. The reusable adapter preserves the exact semantic prompt contract across those provider interfaces.


In [1]:
from collections import Counter
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from time import perf_counter
import json
import os
import sqlite3

from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv
from openai import OpenAI

from domain_adapt.context_engineering import (
    load_context_engineering_config,
    request_frontier_context_engineered_sql,
)
from domain_adapt.evaluation import compare_sql_results
from domain_adapt.evaluation_analysis import (
    compare_runs,
    failure_category,
    index_results,
    summarize_results,
)
from domain_adapt.finch import make_reference_compatible


def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root")


def sha256_file(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_crlf_text(path: Path) -> str:
    # Existing config hashes were recorded on Windows. Canonical newline
    # handling keeps validation stable on Windows and Linux.
    canonical = path.read_text(encoding="utf-8").replace("\r\n", "\n")
    canonical = canonical.replace("\r", "\n").replace("\n", "\r\n")
    return sha256(canonical.encode("utf-8")).hexdigest()


def save_json_atomic(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(value, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary_path.replace(path)


REPOSITORY_ROOT = find_repository_root(Path.cwd())
DATA_DIR = REPOSITORY_ROOT / "data"
RUN_DIR = DATA_DIR / "runs"
SEALED_RUN_DIR = RUN_DIR / "sealed"

DB_PATH = DATA_DIR / "financial.sqlite"
DATASET_PATH = DATA_DIR / "finch_dataset.json"
MANIFEST_PATH = REPOSITORY_ROOT / "configs" / "eval_manifest_v1.json"
FRONTIER_CONTEXT_CONFIG_PATH = (
    REPOSITORY_ROOT / "configs" / "frontier_context_engineered_v1.json"
)
CONTEXT_CONTRACT_PATH = (
    REPOSITORY_ROOT / "configs" / "slm_context_engineered_v1.json"
)

EXPECTED_FRONTIER_CONTEXT_CONFIG_SHA256 = (
    "026aada549f40ec10fca7b8319fc2ed06a4d99da16d92d200548cd7637711702"
)
EXPECTED_CONTEXT_CONTRACT_SHA256 = (
    "574a775a2c91826a4d0073f78e84c40c3c4f5b6fbb61f656bd63f307252cdf68"
)
EXPECTED_INSTRUCTIONS_SHA256 = (
    "175e0489882789131bba788ad087f79e24a2207460d934778a16e04d7f265509"
)
EXPECTED_SCHEMA_CONTEXT_SHA256 = (
    "634503b9d904730790555dfeabb2a81a920293a9d747416a04e372e122d7b987"
)

load_dotenv(REPOSITORY_ROOT / ".env", override=True)
print("Repository root:", REPOSITORY_ROOT)


Repository root: C:\Users\shchitt\Downloads\Projects\domain-adapt


## 1. Validate both layers of the contract

The small frontier-specific configuration declares the model, provider API, deployment variable, and immutable reference to the selected context contract. The referenced context file contains the actual instructions and schema text.

This avoids copying thousands of characters into a second configuration while still making the GPT-5.2 experiment independently auditable.


In [3]:
for required_path in (
    DB_PATH,
    DATASET_PATH,
    MANIFEST_PATH,
    FRONTIER_CONTEXT_CONFIG_PATH,
    CONTEXT_CONTRACT_PATH,
):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

assert sha256_crlf_text(FRONTIER_CONTEXT_CONFIG_PATH) == (
    EXPECTED_FRONTIER_CONTEXT_CONFIG_SHA256
)
assert sha256_crlf_text(CONTEXT_CONTRACT_PATH) == (
    EXPECTED_CONTEXT_CONTRACT_SHA256
)

frontier_context_config = json.loads(
    FRONTIER_CONTEXT_CONFIG_PATH.read_text(encoding="utf-8")
)
context_contract = load_context_engineering_config(CONTEXT_CONTRACT_PATH)

assert frontier_context_config["api"] == "responses"
assert frontier_context_config["deployment_env"] == (
    "AZURE_AI_MODEL_DEPLOYMENT_NAME"
)
assert frontier_context_config["context_contract_sha256"] == (
    EXPECTED_CONTEXT_CONTRACT_SHA256
)
assert frontier_context_config["generation"]["attempts_per_question"] == 1
assert frontier_context_config["generation"]["candidate_sql_repair"] is False
assert frontier_context_config["generation"]["retry_incorrect_answers"] is False

assert context_contract["instructions_sha256"] == (
    EXPECTED_INSTRUCTIONS_SHA256
)
assert context_contract["schema_context_sha256"] == (
    EXPECTED_SCHEMA_CONTEXT_SHA256
)

print("Frontier wrapper contract validated.")
print("Context contract validated.")
print("No model request made.")


Frontier wrapper contract validated.
Context contract validated.
No model request made.


## 2. Reconstruct the leakage-safe evaluation sets

The 12-question `working_dev` set provides a visible technical check. The 24-question frozen set supplies the fixed benchmark result. The notebook validates membership and disjointness before opening the database.


In [4]:
records = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

working_ids = set(manifest["selection"]["working_dev"]["ids"])
frozen_ids = set(manifest["selection"]["frozen_eval"]["ids"])
assert working_ids.isdisjoint(frozen_ids)


def select_examples(question_ids: set[int]) -> list[dict]:
    return sorted(
        [
            row for row in records
            if row["db_name"] == "bird"
            and row["db_id"] == "financial"
            and row["question_id"] in question_ids
        ],
        key=lambda row: row["question_id"],
    )


working_examples = select_examples(working_ids)
frozen_examples = select_examples(frozen_ids)

assert len(working_examples) == 12
assert len(frozen_examples) == 24
assert {row["question_id"] for row in working_examples} == working_ids
assert {row["question_id"] for row in frozen_examples} == frozen_ids

conn = sqlite3.connect(
    f"file:{DB_PATH.as_posix()}?mode=ro",
    uri=True,
)

print("Working questions:", len(working_examples))
print("Frozen questions:", len(frozen_examples))
print("Overlap:", len(working_ids & frozen_ids))


Working questions: 12
Frozen questions: 24
Overlap: 0


## 3. Create the existing Azure OpenAI client

This uses the same endpoint, tenant, credential flow, and GPT-5.2 deployment as the original frontier baseline. No generation parameters are added beyond the existing maximum-output-token limit; provider defaults remain unchanged.


In [5]:
deployment_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"].strip()
tenant_id = os.environ["AZURE_TENANT_ID"].strip()
endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")

if not endpoint.endswith("/openai/v1"):
    endpoint = f"{endpoint}/openai/v1"

credential = AzureCliCredential(tenant_id=tenant_id)
token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default",
)
frontier_client = OpenAI(
    base_url=f"{endpoint}/",
    api_key=token_provider,
)

print("Deployment:", deployment_name)
print("Declared model:", frontier_context_config["declared_underlying_model"])
print("Declared version:", frontier_context_config["declared_model_version"])
print("No model request made.")


Deployment: gpt-5.2
Declared model: gpt-5.2
Declared version: 2025-12-11
No model request made.


## 4. Optional controlled smoke test

Question 16 is already part of `working_dev`, so it can safely confirm that the Responses API adapter, context contract, output normalization, and evaluator compose correctly. Leave the flag disabled after the first successful check.


In [6]:
RUN_QUESTION_16_SMOKE = True

if RUN_QUESTION_16_SMOKE:
    example = next(row for row in working_examples if row["question_id"] == 16)
    response, raw_sql, normalized_sql, was_unwrapped = (
        request_frontier_context_engineered_sql(
            client=frontier_client,
            deployment_name=deployment_name,
            question=example["question"],
            config=context_contract,
        )
    )
    reference_sql, rules = make_reference_compatible(example["SQL"])
    comparison = compare_sql_results(
        conn=conn,
        reference_sql=reference_sql,
        candidate_sql=normalized_sql,
    )
    print("Raw SQL:\n", raw_sql)
    print("Normalized SQL:\n", normalized_sql)
    print("Fence unwrapped:", was_unwrapped)
    print("Passed:", comparison.equivalent)
    print("Reason:", comparison.reason)
    print("Reference-only rules:", rules)
    print("Usage:", response.usage)
else:
    print("Question 16 smoke test skipped.")


Raw SQL:
 SELECT account.date
FROM trans
JOIN account ON account.account_id = trans.account_id
WHERE trans.amount = 840
  AND trans.date = '1998-10-14';
Normalized SQL:
 SELECT account.date
FROM trans
JOIN account ON account.account_id = trans.account_id
WHERE trans.amount = 840
  AND trans.date = '1998-10-14';
Fence unwrapped: False
Passed: True
Reason: unordered results match
Reference-only rules: ['case_insensitive_text_equality']
Usage: ResponseUsage(input_tokens=1265, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=42, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1307)


## 5. Reusable, resumable collection function

The same collector is used for both splits. It makes one request, applies strict fence normalization, evaluates the SQL, records provider metadata, and saves atomically before continuing.

For `working_dev`, IDs and outcomes are visible. For the frozen benchmark, only progress counts are printed.


In [7]:
def collect_results(
    *,
    run: dict,
    examples: list[dict],
    output_path: Path,
    reveal_outcomes: bool,
) -> None:
    completed_ids = set(index_results(run["results"]))
    pending_examples = [
        example
        for example in examples
        if example["question_id"] not in completed_ids
    ]

    for example in pending_examples:
        question_id = example["question_id"]
        started = perf_counter()

        try:
            response, raw_sql, normalized_sql, was_unwrapped = (
                request_frontier_context_engineered_sql(
                    client=frontier_client,
                    deployment_name=deployment_name,
                    question=example["question"],
                    config=context_contract,
                )
            )
            reference_sql, rules = make_reference_compatible(example["SQL"])
            comparison = compare_sql_results(
                conn=conn,
                reference_sql=reference_sql,
                candidate_sql=normalized_sql,
            )
            result = {
                "question_id": question_id,
                "difficulty": example["difficulty"],
                "raw_generated_sql": raw_sql,
                "generated_sql": normalized_sql,
                "output_envelope_normalized": was_unwrapped,
                "passed": comparison.equivalent,
                "reason": comparison.reason,
                "expected_row_count": comparison.expected_row_count,
                "candidate_row_count": comparison.candidate_row_count,
                "reference_compatibility_rules": rules,
                "response_id": response.id,
                "response_status": response.status,
                "response_model": response.model,
                "usage": response.usage.model_dump() if response.usage else {},
                "api_error": None,
            }
        except Exception as error:
            result = {
                "question_id": question_id,
                "difficulty": example["difficulty"],
                "raw_generated_sql": None,
                "generated_sql": None,
                "output_envelope_normalized": False,
                "passed": False,
                "reason": "model request failed",
                "expected_row_count": None,
                "candidate_row_count": None,
                "reference_compatibility_rules": None,
                "response_id": None,
                "response_status": None,
                "response_model": None,
                "usage": {},
                "api_error": f"{type(error).__name__}: {error}",
            }

        result["latency_seconds"] = round(perf_counter() - started, 3)
        run["results"].append(result)
        index_results(run["results"])
        save_json_atomic(output_path, run)

        if reveal_outcomes:
            outcome = "PASS" if result["passed"] else "FAIL"
            print(
                f"[{len(run['results']):02d}/{len(examples):02d}] "
                f"ID {question_id:>3} ({example['difficulty']:<6}) "
                f"{outcome}: {result['reason']}"
            )
        else:
            print(f"Completed {len(run['results'])}/{len(examples)}")


## 6. Run the 12-question working check

The working result is diagnostic, not a prompt-selection opportunity. Do not alter the fixed context after seeing it. The artifact is resumable and rejects contract drift or duplicate IDs.


In [8]:
WORKING_RUN_PATH = (
    RUN_DIR / "gpt_5_2_context_engineered_working_dev_v1.json"
)
RUN_WORKING_EVALUATION = True

base_run_contract = {
    "benchmark": manifest["benchmark"],
    "variant": "gpt-5.2-context-engineered-v1",
    "deployment_name": deployment_name,
    "declared_underlying_model": (
        frontier_context_config["declared_underlying_model"]
    ),
    "declared_model_version": (
        frontier_context_config["declared_model_version"]
    ),
    "frontier_context_config_sha256": (
        EXPECTED_FRONTIER_CONTEXT_CONFIG_SHA256
    ),
    "context_contract_sha256": EXPECTED_CONTEXT_CONTRACT_SHA256,
    "instructions_sha256": EXPECTED_INSTRUCTIONS_SHA256,
    "schema_context_sha256": EXPECTED_SCHEMA_CONTEXT_SHA256,
    "attempts_per_question": 1,
    "candidate_sql_repair": False,
    "retry_incorrect_answers": False,
    "output_normalization": context_contract["output_normalization"],
}

working_contract = {
    **base_run_contract,
    "run_version": 1,
    "split": "working_dev",
}

if WORKING_RUN_PATH.exists():
    working_run = json.loads(WORKING_RUN_PATH.read_text(encoding="utf-8"))
    for key, expected_value in working_contract.items():
        assert working_run[key] == expected_value
    index_results(working_run["results"])
    print("Resuming working results:", len(working_run["results"]))
elif RUN_WORKING_EVALUATION:
    working_run = {
        **working_contract,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "results": [],
    }
    save_json_atomic(WORKING_RUN_PATH, working_run)
else:
    working_run = None
    print("Working evaluation disabled; no local artifact found.")

if RUN_WORKING_EVALUATION:
    assert working_run is not None
    collect_results(
        run=working_run,
        examples=working_examples,
        output_path=WORKING_RUN_PATH,
        reveal_outcomes=True,
    )
else:
    print("No working model requests made.")


[01/12] ID   5 (medium) PASS: unordered results match
[02/12] ID  11 (medium) PASS: ordered results match
[03/12] ID  14 (easy  ) FAIL: unordered results differ
[04/12] ID  16 (easy  ) PASS: unordered results match
[05/12] ID  45 (easy  ) FAIL: ordered results differ
[06/12] ID  49 (medium) FAIL: unordered results differ
[07/12] ID  61 (hard  ) FAIL: unordered results differ
[08/12] ID  69 (easy  ) PASS: unordered results match
[09/12] ID  81 (hard  ) FAIL: unordered results differ
[10/12] ID  89 (easy  ) PASS: unordered results match
[11/12] ID  99 (easy  ) PASS: unordered results match
[12/12] ID 105 (medium) FAIL: column count differs: 3 != 2


In [9]:
if WORKING_RUN_PATH.exists():
    working_run = json.loads(WORKING_RUN_PATH.read_text(encoding="utf-8"))
    working_results = working_run["results"]
    assert len(working_results) == 12
    assert set(index_results(working_results)) == working_ids
    working_summary = summarize_results(working_results)
    working_run["summary"] = working_summary
    save_json_atomic(WORKING_RUN_PATH, working_run)

    print(
        f"Overall: {working_summary['passed']}/12 "
        f"({working_summary['accuracy']:.1%})"
    )
    print(
        f"Executable: {working_summary['executable']}/12 "
        f"({working_summary['executable_rate']:.1%})"
    )
    print("By difficulty:", working_summary["by_difficulty"])
    print("Failure categories:", working_summary["failure_categories"])
    print("API errors:", working_summary["api_errors"])
    print("Token usage:", working_summary["token_usage"])
    print("Median latency:", working_summary["median_latency_seconds"])
    print("Artifact SHA-256:", sha256_file(WORKING_RUN_PATH))
else:
    working_summary = None
    print("Working summary unavailable.")


Overall: 6/12 (50.0%)
Executable: 12/12 (100.0%)
By difficulty: {'easy': {'passed': 4, 'total': 6, 'accuracy': 0.6666666666666666}, 'medium': {'passed': 2, 'total': 4, 'accuracy': 0.5}, 'hard': {'passed': 0, 'total': 2, 'accuracy': 0.0}}
Failure categories: {'unordered_results_differ': 4, 'ordered_results_differ': 1, 'column_count_differs': 1}
API errors: 0
Token usage: {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 15899}
Median latency: 1.5154999999999998
Artifact SHA-256: 6661989ed19f8dc8ab6ed6f719f47e7b931b25b9f56ba8aa672681d971f2e741


## 7. Run the fixed 24-question benchmark

This benchmark uses the same already-established frozen IDs. Because earlier variants have already opened the benchmark, this is best described as an **additional fixed-system benchmark**, not a newly untouched test.

During collection, question IDs and outcomes remain hidden. Set `RUN_FROZEN_EVALUATION = True` only after the working path is technically healthy. The run is resumable and saves after every request.


In [10]:
FROZEN_RUN_PATH = (
    SEALED_RUN_DIR / "gpt_5_2_context_engineered_frozen_eval_v1.json"
)
RUN_FROZEN_EVALUATION = True

frozen_contract = {
    **base_run_contract,
    "run_version": 1,
    "split": "frozen_eval",
}

if FROZEN_RUN_PATH.exists():
    frozen_run = json.loads(FROZEN_RUN_PATH.read_text(encoding="utf-8"))
    for key, expected_value in frozen_contract.items():
        assert frozen_run[key] == expected_value
    index_results(frozen_run["results"])
    print("Resuming frozen results:", len(frozen_run["results"]))
elif RUN_FROZEN_EVALUATION:
    frozen_run = {
        **frozen_contract,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "results": [],
    }
    save_json_atomic(FROZEN_RUN_PATH, frozen_run)
else:
    frozen_run = None
    print("Frozen evaluation disabled; no local artifact found.")

if RUN_FROZEN_EVALUATION:
    assert frozen_run is not None
    collect_results(
        run=frozen_run,
        examples=frozen_examples,
        output_path=FROZEN_RUN_PATH,
        reveal_outcomes=False,
    )
else:
    print("No frozen model requests made.")


Completed 1/24
Completed 2/24
Completed 3/24
Completed 4/24
Completed 5/24
Completed 6/24
Completed 7/24
Completed 8/24
Completed 9/24
Completed 10/24
Completed 11/24
Completed 12/24
Completed 13/24
Completed 14/24
Completed 15/24
Completed 16/24
Completed 17/24
Completed 18/24
Completed 19/24
Completed 20/24
Completed 21/24
Completed 22/24
Completed 23/24
Completed 24/24


## 8. Infrastructure-only completion

Correctness retries are forbidden. If an authentication or infrastructure failure produced no response ID, no SQL, and no usage, the missing request may be completed once without discarding any valid responses.

Set `COMPLETE_INFRASTRUCTURE_FAILURES = True` only when the diagnostic cell reports such missing responses. The resulting v2 artifact preserves every valid v1 result.


In [11]:
COMPLETED_FROZEN_RUN_PATH = (
    SEALED_RUN_DIR / "gpt_5_2_context_engineered_frozen_eval_v2.json"
)
COMPLETE_INFRASTRUCTURE_FAILURES = False


def is_infrastructure_only_failure(result: dict) -> bool:
    return (
        result.get("api_error") is not None
        and result.get("response_id") is None
        and result.get("generated_sql") is None
        and not result.get("usage")
    )


if FROZEN_RUN_PATH.exists():
    initial_frozen_run = json.loads(
        FROZEN_RUN_PATH.read_text(encoding="utf-8")
    )
    infrastructure_failures = [
        result
        for result in initial_frozen_run["results"]
        if is_infrastructure_only_failure(result)
    ]
    valid_frozen_results = [
        result
        for result in initial_frozen_run["results"]
        if not is_infrastructure_only_failure(result)
    ]
    print("Valid responses:", len(valid_frozen_results))
    print("Infrastructure-missing responses:", len(infrastructure_failures))
else:
    initial_frozen_run = None
    infrastructure_failures = []
    valid_frozen_results = []
    print("Initial frozen artifact unavailable.")


Valid responses: 24
Infrastructure-missing responses: 0


In [12]:
if COMPLETE_INFRASTRUCTURE_FAILURES:
    assert initial_frozen_run is not None
    assert infrastructure_failures

    if COMPLETED_FROZEN_RUN_PATH.exists():
        completed_frozen_run = json.loads(
            COMPLETED_FROZEN_RUN_PATH.read_text(encoding="utf-8")
        )
        assert completed_frozen_run["source_run_sha256"] == sha256_file(
            FROZEN_RUN_PATH
        )
        index_results(completed_frozen_run["results"])
    else:
        completed_frozen_run = {
            **{
                key: value
                for key, value in initial_frozen_run.items()
                if key not in {"results", "summary"}
            },
            "run_version": 2,
            "source_run": FROZEN_RUN_PATH.name,
            "source_run_sha256": sha256_file(FROZEN_RUN_PATH),
            "completion_policy": (
                "Repeat only requests that received no model response "
                "because of an infrastructure failure."
            ),
            "results": valid_frozen_results,
        }
        save_json_atomic(COMPLETED_FROZEN_RUN_PATH, completed_frozen_run)

    missing_ids = {
        result["question_id"] for result in infrastructure_failures
    }
    completed_ids = set(index_results(completed_frozen_run["results"]))
    completion_examples = [
        example
        for example in frozen_examples
        if example["question_id"] in missing_ids
        and example["question_id"] not in completed_ids
    ]
    collect_results(
        run=completed_frozen_run,
        examples=completion_examples,
        output_path=COMPLETED_FROZEN_RUN_PATH,
        reveal_outcomes=False,
    )
else:
    print("Infrastructure completion skipped.")


Infrastructure completion skipped.


## 9. Select and summarize the official frontier-context run

The v2 artifact is used only if infrastructure completion was necessary. The official artifact must contain all 24 unique frozen IDs and zero API errors.


In [13]:
official_frozen_path = None
official_frozen_run = None

for candidate_path in (COMPLETED_FROZEN_RUN_PATH, FROZEN_RUN_PATH):
    if not candidate_path.exists():
        continue
    candidate_run = json.loads(candidate_path.read_text(encoding="utf-8"))
    candidate_results = candidate_run["results"]
    if (
        len(candidate_results) == 24
        and set(index_results(candidate_results)) == frozen_ids
        and all(result.get("api_error") is None for result in candidate_results)
    ):
        official_frozen_path = candidate_path
        official_frozen_run = candidate_run
        break

if official_frozen_run is None:
    frozen_summary = None
    official_frozen_sha256 = None
    print("Official frozen frontier-context run is not complete.")
else:
    frozen_summary = summarize_results(official_frozen_run["results"])
    official_frozen_run["summary"] = frozen_summary
    save_json_atomic(official_frozen_path, official_frozen_run)
    official_frozen_sha256 = sha256_file(official_frozen_path)

    print("Official GPT-5.2 context-engineered evaluation complete.")
    print(
        f"Overall: {frozen_summary['passed']}/24 "
        f"({frozen_summary['accuracy']:.1%})"
    )
    print(
        f"Executable: {frozen_summary['executable']}/24 "
        f"({frozen_summary['executable_rate']:.1%})"
    )
    print("By difficulty:", frozen_summary["by_difficulty"])
    print("Failure categories:", frozen_summary["failure_categories"])
    print("API errors:", frozen_summary["api_errors"])
    print("Token usage:", frozen_summary["token_usage"])
    print("Median latency:", frozen_summary["median_latency_seconds"])
    print("Sealed-run SHA-256:", official_frozen_sha256)


Official GPT-5.2 context-engineered evaluation complete.
Overall: 16/24 (66.7%)
Executable: 24/24 (100.0%)
By difficulty: {'easy': {'passed': 10, 'total': 12, 'accuracy': 0.8333333333333334}, 'medium': {'passed': 5, 'total': 9, 'accuracy': 0.5555555555555556}, 'hard': {'passed': 1, 'total': 3, 'accuracy': 0.3333333333333333}}
Failure categories: {'ordered_results_differ': 3, 'unordered_results_differ': 5}
API errors: 0
Token usage: {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 32117}
Median latency: 1.523
Sealed-run SHA-256: 2048f1e584e4f58505e4397bebd823aa250c97e2fa2515fbefd43aa9fe102bd3


## 10. Measure the incremental value of context for GPT-5.2

This is the primary comparison: the model, deployment, API, database, and questions are held constant while the minimal schema prompt is replaced by the selected context-engineered system.

The transition analysis distinguishes net accuracy from churn—new passes, regressions, and unchanged outcomes.


In [14]:
MINIMAL_WORKING_PATH = RUN_DIR / "gpt_5_2_minimal_working_dev_v1.json"
MINIMAL_FROZEN_PATH = SEALED_RUN_DIR / "gpt_5_2_minimal_frozen_eval_v1.json"


def compare_pair(label: str, minimal_path: Path, context_path: Path) -> dict | None:
    if not minimal_path.exists() or not context_path.exists():
        print(f"{label}: comparison artifacts unavailable")
        return None

    minimal_run = json.loads(minimal_path.read_text(encoding="utf-8"))
    context_run = json.loads(context_path.read_text(encoding="utf-8"))
    minimal_summary = summarize_results(minimal_run["results"])
    context_summary = summarize_results(context_run["results"])
    transitions = compare_runs(minimal_run["results"], context_run["results"])
    transition_counts = Counter(row["transition"] for row in transitions)

    comparison = {
        "minimal_summary": minimal_summary,
        "context_summary": context_summary,
        "accuracy_delta": (
            context_summary["accuracy"] - minimal_summary["accuracy"]
        ),
        "executable_rate_delta": (
            context_summary["executable_rate"]
            - minimal_summary["executable_rate"]
        ),
        "total_token_delta": (
            context_summary["token_usage"]["total_tokens"]
            - minimal_summary["token_usage"]["total_tokens"]
        ),
        "transition_counts": dict(transition_counts),
    }

    print(f"{label}:")
    print(
        "  Accuracy:",
        f"{minimal_summary['passed']}/{minimal_summary['total']} -> "
        f"{context_summary['passed']}/{context_summary['total']}",
    )
    print(
        "  Executable:",
        f"{minimal_summary['executable']}/{minimal_summary['total']} -> "
        f"{context_summary['executable']}/{context_summary['total']}",
    )
    print("  Accuracy delta:", f"{comparison['accuracy_delta']:+.1%}")
    print("  Executable delta:", f"{comparison['executable_rate_delta']:+.1%}")
    print("  Token delta:", f"{comparison['total_token_delta']:+,}")
    print("  Outcome transitions:", comparison["transition_counts"])
    return comparison


working_comparison = compare_pair(
    "Working set",
    MINIMAL_WORKING_PATH,
    WORKING_RUN_PATH,
)

frozen_comparison = (
    compare_pair(
        "Frozen benchmark",
        MINIMAL_FROZEN_PATH,
        official_frozen_path,
    )
    if official_frozen_path is not None
    else None
)


Working set:
  Accuracy: 4/12 -> 6/12
  Executable: 12/12 -> 12/12
  Accuracy delta: +16.7%
  Executable delta: +0.0%
  Token delta: +11,389
  Outcome transitions: {'still_passes': 4, 'improved': 2, 'still_fails': 6}
Frozen benchmark:
  Accuracy: 11/24 -> 16/24
  Executable: 24/24 -> 24/24
  Accuracy delta: +20.8%
  Executable delta: +0.0%
  Token delta: +23,036
  Outcome transitions: {'improved': 5, 'still_fails': 8, 'still_passes': 11}


## 11. Complete benchmark matrix

The final table compares model capability, system context, and weight adaptation. Total tokens measure traffic, not cost: different models and deployment types have different prices.


In [15]:
FROZEN_VARIANT_PATHS = {
    "GPT-5.2 + minimal context": MINIMAL_FROZEN_PATH,
    "GPT-5.2 + engineered context": official_frozen_path,
    "Ministral-3B + minimal context": (
        SEALED_RUN_DIR / "ministral_3b_minimal_frozen_eval_v1.json"
    ),
    "Ministral-3B + engineered context": (
        SEALED_RUN_DIR
        / "ministral_3b_context_engineered_frozen_eval_v2.json"
    ),
    "SFT v1 + engineered context": (
        SEALED_RUN_DIR / "ministral_3b_sft_v1_frozen_eval_v1.json"
    ),
}

variant_summaries = {}
for name, path in FROZEN_VARIANT_PATHS.items():
    if path is None or not path.exists():
        continue
    run = json.loads(path.read_text(encoding="utf-8"))
    variant_summaries[name] = summarize_results(run["results"])

print(
    f"{'Variant':<40} {'Accuracy':>10} "
    f"{'Executable':>12} {'Tokens':>10} {'Latency':>9}"
)
print("-" * 87)
for name, summary in variant_summaries.items():
    latency = summary["median_latency_seconds"]
    latency_text = "n/a" if latency is None else f"{latency:.3f}s"
    print(
        f"{name:<40} "
        f"{summary['passed']}/{summary['total']:<7} "
        f"{summary['executable']}/{summary['total']:<9} "
        f"{summary['token_usage']['total_tokens']:>10,} "
        f"{latency_text:>9}"
    )


Variant                                    Accuracy   Executable     Tokens   Latency
---------------------------------------------------------------------------------------
GPT-5.2 + minimal context                11/24      24/24             9,081    1.827s
GPT-5.2 + engineered context             16/24      24/24            32,117    1.523s
Ministral-3B + minimal context           1/24      10/24             8,254    0.940s
Ministral-3B + engineered context        7/24      17/24            34,241    1.079s
SFT v1 + engineered context              0/24      0/24                 0    0.140s


## 12. Inspect failure-category movement without retuning

This deterministic analysis explains *how* GPT-5.2 changed under the engineered context. It is diagnostic only. The frozen failures must not trigger another context-selection cycle.


In [16]:
if official_frozen_run is not None and MINIMAL_FROZEN_PATH.exists():
    minimal_frozen_run = json.loads(
        MINIMAL_FROZEN_PATH.read_text(encoding="utf-8")
    )
    minimal_by_id = index_results(minimal_frozen_run["results"])
    context_by_id = index_results(official_frozen_run["results"])

    residual_transitions = Counter()
    for question_id in sorted(frozen_ids):
        minimal = minimal_by_id[question_id]
        engineered = context_by_id[question_id]
        if minimal["passed"] or engineered["passed"]:
            continue
        residual_transitions[
            (
                failure_category(minimal),
                failure_category(engineered),
            )
        ] += 1

    print("Residual failure transitions:")
    for transition, count in residual_transitions.most_common():
        print(f"- {transition[0]} -> {transition[1]}: {count}")
else:
    print("Failure-transition analysis is not ready.")


Residual failure transitions:
- unordered_results_differ -> unordered_results_differ: 4
- ordered_results_differ -> ordered_results_differ: 2
- column_count_differs -> ordered_results_differ: 1
- column_count_differs -> unordered_results_differ: 1


## 13. Record the architectural conclusion

The decision artifact captures the fixed comparison for Milestone 10. Context is considered beneficial for GPT-5.2 when frozen accuracy improves; executability, latency, token traffic, and regressions determine whether that gain is operationally attractive.


In [17]:
DECISION_PATH = RUN_DIR / "frontier_context_comparison_v1.json"

if frozen_comparison is not None:
    accuracy_delta = frozen_comparison["accuracy_delta"]
    if accuracy_delta > 0:
        conclusion = (
            "The selected context improved GPT-5.2 frozen accuracy. "
            "Context engineering and frontier capability are complementary."
        )
    elif accuracy_delta == 0:
        conclusion = (
            "The selected context did not change GPT-5.2 frozen accuracy. "
            "Most supplied information was already recoverable by the model."
        )
    else:
        conclusion = (
            "The selected context reduced GPT-5.2 frozen accuracy. "
            "The additional context or guardrails distracted the stronger model."
        )

    decision_record = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "experiment": "gpt-5.2-context-engineering-ablation-v1",
        "working_comparison": working_comparison,
        "frozen_comparison": frozen_comparison,
        "official_frozen_artifact": official_frozen_path.name,
        "official_frozen_sha256": official_frozen_sha256,
        "conclusion": conclusion,
    }
    save_json_atomic(DECISION_PATH, decision_record)

    print("Conclusion:", conclusion)
    print("Decision artifact:", DECISION_PATH)
else:
    print("Conclusion deferred until the fixed benchmark is complete.")


Conclusion: The selected context improved GPT-5.2 frozen accuracy. Context engineering and frontier capability are complementary.
Decision artifact: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\frontier_context_comparison_v1.json


## 14. Handoff to Milestone 10

After running the notebook, add the printed working result, frozen result, comparison deltas, and conclusion to this final narrative before committing.

Milestone 10 can then answer the DomainAdapt research question with a complete matrix:

- how much capability came from the stronger model;
- how much came from system context;
- whether context and frontier capability were complementary;
- whether SFT added anything beyond the selected system; and
- which architecture offers the best accuracy, reliability, latency, token, cost, and maintenance trade-off.

Do not optimize the context from the frozen GPT-5.2 failures. This notebook completes the planned V1 evidence; it does not reopen model or prompt selection.
